# 01 · From Markdown to a grounded answer

**Question** — If Cutting gets faster and Assembly stays at its current capacity, does the line produce more finished units per hour?

This lab uses three original notes about a fictional production line. Start with the retrieved evidence so you can see what the model would have to work with. The final cell can send that evidence to an Ollama model you choose.


## 1. Load and partition the notes

`partition_md()` reads the Markdown and returns typed elements such as headings and paragraphs. Inspect a few to see what the parser recognized before you chunk the documents.


In [ ]:
from pathlib import Path
from uuid import uuid4

import chromadb
from unstructured.chunking.title import chunk_by_title
from unstructured.partition.md import partition_md

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

corpus_dir = project_root / "data" / "sample"
paths = sorted(corpus_dir.glob("*.md"))
if not paths:
    raise FileNotFoundError(f"No Markdown files in {corpus_dir}. Open the repo in VS Code.")

elements_by_path = {path: partition_md(filename=str(path)) for path in paths}

for path, elements in elements_by_path.items():
    print(f"{path.name} — {len(elements)} elements")
for element in elements_by_path[paths[0]][:3]:
    print(f"{element.category} | {element.text}")


## 2. Chunk by title and keep the source

Chunk each file separately so a passage has a clear source. Keep the latest detected heading with each chunk, including continuation chunks. A source path and heading act like a file and section reference.


In [ ]:
records = []

for path, elements in elements_by_path.items():
    source = path.relative_to(project_root).as_posix()
    heading = path.stem
    chunks = chunk_by_title(
        elements,
        max_characters=700,
        combine_text_under_n_chars=0,
        include_orig_elements=True,
    )

    for number, chunk in enumerate(chunks, start=1):
        heading = next(
            (
                element.text
                for element in (chunk.metadata.orig_elements or [])
                if element.category == "Title"
            ),
            heading,
        )
        records.append(
            {
                "id": f"{source}::{number}",
                "text": chunk.text,
                "metadata": {"source": source, "heading": heading},
            }
        )

print(f"Prepared {len(records)} chunks from {len(paths)} notes.")
print(records[0])


## 3. Embed the chunks and index them in Chroma

Chroma uses its default local embedding model to encode the chunk text. The first run may download the model. This notebook builds a fresh in-memory collection whenever you rerun the indexing cell.


In [ ]:
client = chromadb.EphemeralClient()
collection = client.create_collection(
    name=f"engineering-lab-{uuid4().hex[:8]}",
    configuration={"hnsw": {"space": "cosine"}},
)
collection.add(
    ids=[record["id"] for record in records],
    documents=[record["text"] for record in records],
    metadatas=[record["metadata"] for record in records],
)
print(f"Indexed {collection.count()} chunks.")


## 4. Ask a question and inspect the passages

Chroma embeds the question with the same model and returns nearby chunks. A smaller cosine distance indicates a closer vector match. Read the passages to see whether they actually contain the facts you need.


In [ ]:
def retrieve(question, *, source=None, k=3):
    result = collection.query(
        query_texts=[question],
        n_results=k,
        where={"source": source} if source is not None else None,
        include=["documents", "metadatas", "distances"],
    )
    return [
        {"id": chunk_id, "text": text, "metadata": metadata, "distance": distance}
        for chunk_id, text, metadata, distance in zip(
            result["ids"][0],
            result["documents"][0],
            result["metadatas"][0],
            result["distances"][0],
            strict=True,
        )
    ]


def format_evidence(matches):
    passages = []
    for number, match in enumerate(matches, start=1):
        source = match["metadata"]["source"]
        heading = match["metadata"]["heading"]
        passages.append(f"[{number}] {source} > {heading}\n{match['text']}")
    return "\n\n".join(passages)


question = "If cutting gets faster but assembly stays the same, does the line produce more per hour?"
evidence = retrieve(question)

print(question, end="\n\n")
print(format_evidence(evidence))
print("Cosine distances", [round(match["distance"], 3) for match in evidence])


## 5. Repeat the search with a source filter

Compare the full-corpus results with a search restricted to the bottleneck note. The source filter decides which chunks are eligible. Vector similarity ranks the eligible chunks.


In [ ]:
bottleneck_source = "data/sample/01-bottleneck-throughput.md"
filtered_evidence = retrieve(question, source=bottleneck_source, k=2)
print(format_evidence(filtered_evidence))


## 6. Optional · Generate a cited answer with Ollama

Set `ollama_model` to an installed model name on a running Ollama instance, then run the cell. The model gets the question and numbered passages from Step 5 through `filtered_evidence`. To try the full-corpus search, pass `evidence` instead. For a remote Ollama instance, set `ollama_base_url` to its reachable API address on port `11434`.

**Check the answer** — Assembly still caps the line at **40 finished units per hour**. The supporting passage is **Faster cutting with the same bottleneck** in `01-bottleneck-throughput.md`. Check that the model's citation number points to that passage. Try an unsupported question, such as “What is the voltage of the warehouse forklift battery?”, to see how the model handles missing evidence.


In [ ]:
import json
from urllib.request import Request, urlopen

ollama_model = ""  # Set to an installed model, e.g. "qwen2.5-coder:14b"
ollama_base_url = "http://127.0.0.1:11434"


def answer_with_ollama(question, matches, *, model, base_url):
    if not matches:
        raise ValueError("No passages reached Section 6. Rerun Section 5.")

    system_prompt = """Answer using the numbered source passages.
You may combine facts from different sentences and draw direct conclusions.
For a comparison, state each value with its units, then give your conclusion.
Put the supporting passage number in square brackets after each factual claim.
If a fact needed to answer is missing, name that fact."""

    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": f"Question\n{question}\n\nSource passages\n{format_evidence(matches)}",
            },
        ],
        "stream": False,
        "options": {"temperature": 0},
    }
    request = Request(
        f"{base_url.rstrip('/')}/api/chat",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urlopen(request, timeout=180) as response:
        return json.load(response)["message"]["content"]


if ollama_model:
    print(answer_with_ollama(
        question,
        filtered_evidence,
        model=ollama_model,
        base_url=ollama_base_url,
    ))
else:
    print("Set ollama_model to run answer generation. The retrieved evidence is shown above.")
